### Load csv into pandas dataframe, parse date and time into single timestamp, filter out unimportant columns and low confidence records.

In [ ]:
import pandas as pd
from pathlib import Path

# Insert the date (YYYY-MM-DD) of the VIIRS file you want to process
DATE = "2026-05-02"

if not DATE:
    raise Exception(
        "Date is missing: please insert the date of the VIIRS file you want to process"
    )

file_path = Path.cwd().parent.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{DATE}.csv"
)

data = pd.read_csv(file_path)

data.drop(columns=["satellite", "instrument", "version", "acq_time"], inplace=True)

data = data[data["confidence"] != "l"]

data.info()
data.head()


<class 'pandas.DataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    329115 non-null  float64
 1   longitude   329115 non-null  float64
 2   bright_ti4  329115 non-null  float64
 3   scan        329115 non-null  float64
 4   track       329115 non-null  float64
 5   acq_date    329115 non-null  str    
 6   confidence  329115 non-null  str    
 7   bright_ti5  329115 non-null  float64
 8   frp         329115 non-null  float64
 9   daynight    329115 non-null  str    
dtypes: float64(7), str(3)
memory usage: 27.6 MB


,latitude,longitude,bright_ti4,scan,track,acq_date,confidence,bright_ti5,frp,daynight
0,32.33215,44.09279,306.72,0.71,0.75,2026-05-01,n,288.92,2.80,N
1,32.88856,35.09304,296.63,0.44,0.46,2026-05-01,n,281.80,1.02,N
2,33.15357,44.78751,302.25,0.73,0.76,2026-05-01,n,287.02,2.17,N
3,33.15594,44.77987,316.28,0.73,0.76,2026-05-01,n,288.05,2.17,N
4,33.15751,44.78342,342.93,0.73,0.76,2026-05-01,n,289.27,6.42,N


### Load other data into geodataframe

In [225]:
import geopandas as gpd

fire_data = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.longitude, data.latitude), crs="EPSG:4326"
)

countries_area = pd.read_csv(Path.cwd().parent.joinpath("data/raw", "surface_area.csv"))

countries_boundaries = gpd.read_file(
    Path.cwd().parent.joinpath("data/raw", "geoboundaries_world.geojson")
).to_crs("EPSG:4326")

# check for invalid geometries and repair them if necessary
if not countries_boundaries.is_valid.all():
    countries_boundaries.make_valid()
    print("Repairing invalid geometries")
if not countries_boundaries.is_valid.all():
    raise Exception("Some geometries are invalid and not reparable")
countries_boundaries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   shapeGroup  218 non-null    str     
 1   shapeType   218 non-null    str     
 2   shapeName   218 non-null    str     
 3   geometry    218 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 6.9 KB


### Filter out attributes in country area and polygons, cast strings into correct types
Also keep only area data from the last surveyed year (2023).

In [226]:
countries_area = countries_area[countries_area["TIME_PERIOD"] == 2023]
countries_area = countries_area[["REF_AREA", "OBS_VALUE"]]
countries_area["OBS_VALUE"] = countries_area["OBS_VALUE"].astype(float)
countries_area = countries_area.rename(
    columns={"OBS_VALUE": "country_area", "REF_AREA": "iso_code"}
)
countries_boundaries = countries_boundaries.rename(
    columns={"shapeGroup": "iso_code", "shapeName": "name"}
)
countries_boundaries = countries_boundaries.drop(columns=["shapeType"])
countries_area.dropna()
countries_area.info()

<class 'pandas.DataFrame'>
Index: 215 entries, 1744 to 9791
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   iso_code      215 non-null    str    
 1   country_area  215 non-null    float64
dtypes: float64(1), str(1)
memory usage: 5.0 KB


### Compute true pixel area for fires pixels

In [227]:
fire_data["fire_area"] = fire_data["scan"] * fire_data["track"]
fire_data = fire_data[["acq_date", "geometry", "fire_area"]]
fire_data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   acq_date   329115 non-null  str     
 1   geometry   329115 non-null  geometry
 2   fire_area  329115 non-null  float64 
dtypes: float64(1), geometry(1), str(1)
memory usage: 10.0 MB


### Join countries polygons with area table

In [228]:
countries_boundaries_area = pd.merge(
    countries_boundaries,
    countries_area,
    left_on="iso_code",
    right_on="iso_code",
    how="left",
)
# countries_boundaries_area = countries_boundaries_area.drop(columns="iso_code")
countries_boundaries_area.info()


selected = countries_boundaries_area.loc[
    countries_boundaries_area["country_area"].isna()
]
selected

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      218 non-null    str     
 1   name          218 non-null    str     
 2   geometry      218 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 6.9 KB


,iso_code,name,geometry,country_area
5,ATA,Antarctica,"MULTIPOLYGON (((-60.06171 -79.6813, -60.05473 ...",NaN
94,XKX,Kosovo,"POLYGON ((20.59429 41.87733, 20.5955 41.8765, ...",NaN
168,TWN,Taiwan,"MULTIPOLYGON (((116.71997 20.70818, 116.71902 ...",NaN
179,VAT,Vatican City,"POLYGON ((12.4538 41.90682, 12.45308 41.90668,...",NaN
198,111,Abyei,"POLYGON ((29 9.67356, 29 10.16667, 27.83333 10...",NaN
199,112,Aksai Chin,"MULTIPOLYGON (((78.69839 34.09307, 78.69837 34...",NaN
200,113,CH-IN,"MULTIPOLYGON (((79.70073 30.97073, 79.70088 30...",NaN
201,114,Demchok,"POLYGON ((79.15197 33.18187, 79.15271 33.18106...",NaN
202,115,Dragonja,"MULTIPOLYGON (((13.67648 45.44426, 13.67648 45...",NaN
203,116,Dramana-Shakatoe,"POLYGON ((89.12804 27.61484, 89.12799 27.61484...",NaN


As we can observe the countries with a geometry coming from the the countries boundaries but without an area are those not officially recognized by UN. Since we need to later compute the percentage of the land surface affected by wildfires, we will drop them

In [229]:
countries_boundaries_area = countries_boundaries_area.dropna()
countries_boundaries_area.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 194 entries, 0 to 197
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   geometry      194 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 7.6 KB


### Spatially join countries with fires. Merge and group fire data by date and countries.

In [230]:
fire_data_countries = gpd.sjoin(
    countries_boundaries_area,
    fire_data,
    how="left",
)

# dissolve() times out and crashes the kernel, worked around by using df.groupby() and joining
# back later with the countries geometries on the iso code key
fire_data_by_countries_dates = pd.DataFrame(
    fire_data_countries.groupby(by=["iso_code", "acq_date", "name", "country_area"])[
        "fire_area"
    ]
    .sum()
    .reset_index()
)
fire_data_by_countries = pd.DataFrame(
    fire_data_countries[["iso_code", "name", "country_area", "fire_area"]]
    .groupby(by=["iso_code", "name", "country_area"])["fire_area"]
    .sum()
    .reset_index()
)

fire_data_by_countries = gpd.GeoDataFrame(
    fire_data_by_countries.merge(
        countries_boundaries, on=["iso_code", "name"], how="left"
    )
)

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
dtypes: float64(2), geometry(1), str(2)
memory usage: 7.7 KB


### Compute perentage of wild fires area

In [231]:
fire_data_by_countries["area_perc"] = (
    fire_data_by_countries["fire_area"] / fire_data_by_countries["country_area"]
) * 100

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
 5   area_perc     194 non-null    float64 
dtypes: float64(3), geometry(1), str(2)
memory usage: 9.2 KB


### Simplify polygons
Using the non-simplified polygons created the resulting html map was huge and taking a performance hit. Unlike `simplify()`, `simplify_coverage()` assumes that the GeoSeries forms a polygonal coverage. Polygons borders remain thus consistent.

In [232]:
# TODO optimize memory: write to disk as soon as simplify is done and use del keyword to manually decrease reference count and make garbage collector deallocate the objects

fire_data_by_countries_simplified = fire_data_by_countries.copy()
fire_data_by_countries_simplified["geometry"] = (
    fire_data_by_countries.geometry.simplify_coverage(tolerance=0.05)
)
countries_boundaries_simplified = countries_boundaries.copy()
countries_boundaries_simplified["geometry"] = (
    countries_boundaries.geometry.simplify_coverage(tolerance=0.05)
)

### Save processed data

In [233]:
fire_data_by_countries_simplified.to_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries.gpkg")
)
countries_boundaries_simplified.to_file(
    Path.cwd().parent.joinpath("data/processed", "countries_boundaries_simplified.gpkg")
)